# Exploratory Data Analysis (EDA) on Respiratory Virus Dashboard Dataset

In [1]:
import pandas as pd
from src.enums import measurements, respiratory_virus_keys

rv = pd.read_csv('02-28-respiratory-virus-dashboard-2023-2026.csv')

## Pre-narrow air quality data features
Based on feature descriptions, identify which features are unnecessary, and remove them to focus on relevant features and avoid wasting time.

In [2]:
isBayAreaRegion = rv[respiratory_virus_keys["rpho_region"]] == 'Bay Area' # Only include the same region as the air quality dataset.
isAllAgesAgeGroup = rv[respiratory_virus_keys["age_group"]] == 'All Ages' # Only include all ages data, as it is an aggregate of all age groups and will be easier to stitch with the air quality dataset which is not age-specific.
rv = rv[isBayAreaRegion & isAllAgesAgeGroup]

rv_preview = rv.copy()
rv_preview['null_count'] = rv_preview.isnull().sum(axis=1)
rv_preview.sort_values('null_count', ascending=True).head(3).drop('null_count', axis=1)

,SEASON,AGE_GRP,RPHO_REGION,WEEKENDING,MMWR_WEEK,MMWR_YEAR,COV_POSITIVES,COV_TOTAL_TESTS,COV_TP,COV_TP_LEVEL,...,TOTAL_DEATHS,COV_DEATHS,FLU_DEATHS,RSV_DEATHS,COV_DEATHS_PER,FLU_DEATHS_PER,RSV_DEATHS_PER,SEASON_COV_PED_DEATHS,SEASON_FLU_PED_DEATHS,SEASON_RSV_PED_DEATHS
3923,2025/2026,All Ages,Bay Area,02/21/2026,7,2026,129,20264,0.636597,Very Low,...,216.0,0,2,0,0.000000,0.925926,0.0,NaN,NaN,NaN
3336,2025/2026,All Ages,Bay Area,10/04/2025,40,2025,777,17035,4.561198,Low,...,979.0,11,0,0,1.123596,0.000000,0.0,NaN,NaN,NaN
3322,2025/2026,All Ages,Bay Area,09/27/2025,39,2025,1067,17896,5.962226,Low,...,1027.0,20,0,0,1.947420,0.000000,0.0,NaN,NaN,NaN


## Understanding air quality features
Identify what kind of information is in each column, such as measurement and description.


In [3]:
rv_measurements = {
    respiratory_virus_keys['season']: measurements['nominal'],
    respiratory_virus_keys['age_group']: measurements['nominal'],
    respiratory_virus_keys['rpho_region']: measurements['nominal'],
    respiratory_virus_keys['weekending']: measurements['interval'],
    respiratory_virus_keys['mmwr_week']: measurements['interval'],
    respiratory_virus_keys['mmwr_year']: measurements['interval'],
    respiratory_virus_keys['cov_positives']: measurements['ratio'],
    respiratory_virus_keys['cov_total_tests']: measurements['ratio'],
    respiratory_virus_keys['cov_tp']: measurements['ratio'],
    respiratory_virus_keys['cov_tp_level']: measurements['ordinal'],
    respiratory_virus_keys['flu_positives']: measurements['ratio'],
    respiratory_virus_keys['flu_total_tests']: measurements['ratio'],
    respiratory_virus_keys['flu_tp']: measurements['ratio'],
    respiratory_virus_keys['flu_tp_level']: measurements['ordinal'],
    respiratory_virus_keys['rsv_positives']: measurements['ratio'],
    respiratory_virus_keys['rsv_total_tests']: measurements['ratio'],
    respiratory_virus_keys['rsv_tp']: measurements['ratio'],
    respiratory_virus_keys['rsv_tp_level']: measurements['ordinal'],
    respiratory_virus_keys['flu_a_tests']: measurements['ratio'],
    respiratory_virus_keys['flu_b_tests']: measurements['ratio'],
    respiratory_virus_keys['cov_ed_visits']: measurements['ratio'],
    respiratory_virus_keys['flu_ed_visits']: measurements['ratio'],
    respiratory_virus_keys['rsv_ed_visits']: measurements['ratio'],
    respiratory_virus_keys['pop']: measurements['ratio'],
    respiratory_virus_keys['cov_adm']: measurements['ratio'],
    respiratory_virus_keys['flu_adm']: measurements['ratio'],
    respiratory_virus_keys['rsv_adm']: measurements['ratio'],
    respiratory_virus_keys['cov_adm_rate']: measurements['ratio'],
    respiratory_virus_keys['flu_adm_rate']: measurements['ratio'],
    respiratory_virus_keys['rsv_adm_rate']: measurements['ratio'],
    respiratory_virus_keys['cov_adm_level']: measurements['ordinal'],
    respiratory_virus_keys['flu_adm_level']: measurements['ordinal'],
    respiratory_virus_keys['rsv_adm_level']: measurements['ordinal'],
    respiratory_virus_keys['total_deaths']: measurements['ratio'],
    respiratory_virus_keys['cov_deaths']: measurements['ratio'],
    respiratory_virus_keys['flu_deaths']: measurements['ratio'],
    respiratory_virus_keys['rsv_deaths']: measurements['ratio'],
    respiratory_virus_keys['cov_deaths_per']: measurements['ratio'],
    respiratory_virus_keys['flu_deaths_per']: measurements['ratio'],
    respiratory_virus_keys['rsv_deaths_per']: measurements['ratio'],
    respiratory_virus_keys['season_cov_ped_deaths']: measurements['ratio'],
    respiratory_virus_keys['season_flu_ped_deaths']: measurements['ratio'],
    respiratory_virus_keys['season_rsv_ped_deaths']: measurements['ratio'],
}
rv_descriptions = {
    respiratory_virus_keys['season']: 'The annual surveillance cycle starting in summer (MMWR week 26).',
    respiratory_virus_keys['age_group']: 'Demographic segments: 0-17 years, 18-64 years, and 65+ years.',
    respiratory_virus_keys['rpho_region']: 'The six California Regional Public Health Office areas or the whole state.',
    respiratory_virus_keys['weekending']: 'The Saturday date marking the end of the CDC reporting week.',
    respiratory_virus_keys['mmwr_week']: 'Standardized CDC week number (1 to 52 or 53).',
    respiratory_virus_keys['mmwr_year']: 'The calendar year associated with the reporting week.',
    respiratory_virus_keys['cov_positives']: 'Count of positive COVID-19 lab tests for the week.',
    respiratory_virus_keys['cov_total_tests']: 'Total volume of COVID-19 lab tests performed.',
    respiratory_virus_keys['cov_tp']: 'The percentage of COVID-19 tests that were positive. The formula is (COV_POSITIVES / COV_TOTAL_TESTS) * 100',
    respiratory_virus_keys['cov_tp_level']: 'Positivity level based on 5 seasons of COVID-19 data.',
    respiratory_virus_keys['flu_positives']: 'Count of positive Influenza lab tests for the week.',
    respiratory_virus_keys['flu_total_tests']: 'Total volume of Influenza lab tests performed.',
    respiratory_virus_keys['flu_tp']: 'The percentage of Influenza tests that were positive. The formula is (FLU_POSITIVES / FLU_TOTAL_TESTS) * 100',
    respiratory_virus_keys['flu_tp_level']: 'Positivity level based on 5 seasons of Influenza data.',
    respiratory_virus_keys['rsv_positives']: 'Count of positive RSV lab tests for the week.',
    respiratory_virus_keys['rsv_total_tests']: 'Total volume of RSV lab tests performed.',
    respiratory_virus_keys['rsv_tp']: 'The percentage of RSV tests that were positive. The formula is (RSV_POSITIVES / RSV_TOTAL_TESTS) * 100',
    respiratory_virus_keys['rsv_tp_level']: 'Positivity level based on 5 seasons of RSV data.',
    respiratory_virus_keys['flu_a_tests']: 'Weekly count of positive tests for Influenza Type A.',
    respiratory_virus_keys['flu_b_tests']: 'Weekly count of positive tests for Influenza Type B.',
    respiratory_virus_keys['cov_ed_visits']: 'Percentage of all ER visits related to COVID-19 symptoms.',
    respiratory_virus_keys['flu_ed_visits']: 'Percentage of all ER visits related to Influenza symptoms.',
    respiratory_virus_keys['rsv_ed_visits']: 'Percentage of all ER visits related to RSV symptoms.',
    respiratory_virus_keys['pop']: 'Regional population estimates used for rate calculations.',
    respiratory_virus_keys['cov_adm']: 'New weekly hospital admissions for COVID-19 (NHSN data).',
    respiratory_virus_keys['flu_adm']: 'New weekly hospital admissions for Influenza (NHSN data).',
    respiratory_virus_keys['rsv_adm']: 'New weekly hospital admissions for RSV (NHSN data).',
    respiratory_virus_keys['cov_adm_rate']: 'Weekly rate of COVID-19 admissions based on regional population.',
    respiratory_virus_keys['flu_adm_rate']: 'Weekly rate of Influenza admissions based on regional population.',
    respiratory_virus_keys['rsv_adm_rate']: 'Weekly rate of RSV admissions based on regional population.',
    respiratory_virus_keys['cov_adm_level']: 'Admission level based on 5 seasons of COVID-19 positivity data.',
    respiratory_virus_keys['flu_adm_level']: 'Admission level based on 5 seasons of Influenza positivity data.',
    respiratory_virus_keys['rsv_adm_level']: 'Admission level based on 5 seasons of RSV positivity data.',
    respiratory_virus_keys['total_deaths']: 'Number of all-cause deaths each week.',
    respiratory_virus_keys['cov_deaths']: 'Number of COVID-19 associated deaths each week.',
    respiratory_virus_keys['flu_deaths']: 'Number of influenza-associated deaths each week.',
    respiratory_virus_keys['rsv_deaths']: 'Number of RSV-associated deaths each week.',
    respiratory_virus_keys['cov_deaths_per']: 'Percentage of all-cause deaths attributed to COVID-19.',
    respiratory_virus_keys['flu_deaths_per']: 'Percentage of all-cause deaths attributed to Influenza.',
    respiratory_virus_keys['rsv_deaths_per']: 'Percentage of all-cause deaths attributed to RSV.',
    respiratory_virus_keys['season_cov_ped_deaths']: 'Number of COVID-19 associated pediatric deaths with lab-confirmation each week.',
    respiratory_virus_keys['season_flu_ped_deaths']: 'Number of influenza-associated pediatric deaths with lab-confirmation each week.',
    respiratory_virus_keys['season_rsv_ped_deaths']: 'Number of RSV-associated pediatric deaths with lab-confirmation each week.',
}

rv.info()
pd.DataFrame({
    "Measurement Type": rv_measurements,
    "Description": rv_descriptions,
})

<class 'pandas.core.frame.DataFrame'>
Index: 140 entries, 16 to 3923
Data columns (total 43 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   SEASON                 140 non-null    object 
 1   AGE_GRP                140 non-null    object 
 2   RPHO_REGION            140 non-null    object 
 3   WEEKENDING             140 non-null    object 
 4   MMWR_WEEK              140 non-null    int64  
 5   MMWR_YEAR              140 non-null    int64  
 6   COV_POSITIVES          140 non-null    int64  
 7   COV_TOTAL_TESTS        140 non-null    int64  
 8   COV_TP                 140 non-null    float64
 9   COV_TP_LEVEL           140 non-null    object 
 10  FLU_POSITIVES          140 non-null    int64  
 11  FLU_TOTAL_TESTS        140 non-null    int64  
 12  FLU_TP                 140 non-null    float64
 13  FLU_TP_LEVEL           140 non-null    object 
 14  RSV_POSITIVES          140 non-null    int64  
 15  RSV_TOTAL

,Measurement Type,Description
SEASON,nominal,The annual surveillance cycle starting in summ...
AGE_GRP,nominal,"Demographic segments: 0-17 years, 18-64 years,..."
RPHO_REGION,nominal,The six California Regional Public Health Offi...
WEEKENDING,interval,The Saturday date marking the end of the CDC r...
MMWR_WEEK,interval,Standardized CDC week number (1 to 52 or 53).
MMWR_YEAR,interval,The calendar year associated with the reportin...
COV_POSITIVES,ratio,Count of positive COVID-19 lab tests for the w...
COV_TOTAL_TESTS,ratio,Total volume of COVID-19 lab tests performed.
COV_TP,ratio,The percentage of COVID-19 tests that were pos...
COV_TP_LEVEL,ordinal,Positivity level based on 5 seasons of COVID-1...
